# Vietnamese Emotion Classification - Advanced PhoBERTv2 Fine-tuning

**🌟 Google Colab Ready!** This notebook is optimized for Google Colab with GPU acceleration (20GB RAM recommended).

## 📝 Two Ways to Use This Notebook:

### 🚀 Option 1: Skip Training (Quick Start)
**If you already have a trained model:**
1. Mount Google Drive
2. Set `SKIP_TRAINING = True` in the Quick Start cell
3. Jump to prediction sections

### 🎓 Option 2: Train from Scratch
**For training a new model:**
1. **Upload training data** to Google Drive at: `MyDrive/thesis/data/`
   - Required files: `train_nor_811.xlsx`, `valid_nor_811.xlsx`, `test_nor_811.xlsx`

2. **Enable GPU** in Colab: Runtime → Change runtime type → GPU
   - **A100 (40GB)** - Best performance, batch size 32
   - **V100 (16GB)** - Great performance, batch size 16 (default)
   - **T4 (16GB)** - Good for free tier, reduce batch size to 8 if needed
   - **TPU v2-8** - Also supported but GPU recommended

3. **Run cells in order** - the notebook will:
   - Mount your Google Drive
   - Detect and configure GPU automatically
   - Load training data from Drive
   - Train the model using GPU with mixed precision (FP16)
   - Save the model back to Drive
   - Run predictions with your trained model

---

This notebook implements **state-of-the-art** fine-tuning techniques for BERT-based emotion classification:

## 🚀 Advanced Techniques Implemented:

### 1. **Layer-wise Learning Rate Decay (LLRD)**
- Applies different learning rates to each layer
- Base LR for top layer: **3.5e-6**
- Decay factor: **0.95**
- Classifier head gets **10x** higher LR
- **Preserves pre-trained knowledge** in lower layers

### 2. **Optimized Batch Size & Gradient Accumulation**
- Effective batch size: **16-32** (optimal for BERT)
- Gradient accumulation to simulate larger batches
- **Scheduler steps after EVERY batch** (critical!)
- Stable training with limited memory

### 3. **Early Stopping & Best Model Selection**
- Monitors validation F1 (weighted)
- Patience: 2 epochs
- Saves best model automatically

---

## 📊 Expected Results

**Your Dataset (NEU-ESC):**
- 7 emotions (Enjoyment, Disgust, Other, Sadness, Anger, Fear, Surprise)

**Expected Results:**

| Metric | Expected Range |
|--------|----------------|
| Accuracy | **70-80%** |
| F1 Macro | **50-65%** |
| F1 Weighted | **68-78%** |

---

**Let's get started!**


## 🔧 Google Colab Setup - Mount Google Drive

**Important:** Run this cell first to access your data on Google Drive


In [ ]:
# Mount Google Drive to access training data
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully")
print("✓ Your data should be in: /content/drive/MyDrive/thesis/data")

# Check for TPU and set up if available
import os
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
    print("✓ TPU detected and configured")
except ImportError:
    TPU_AVAILABLE = False
    print("ℹ️  TPU not available, will use GPU/CPU")


## 🚀 Quick Start: Load Pre-trained Model (Skip Training)

**Run this section if you already have a trained model and want to skip training.**

Set `SKIP_TRAINING = True` below to load your saved model and jump to predictions.


In [ ]:
# ============================================================================
# QUICK START: Load existing model and skip training
# ============================================================================

SKIP_TRAINING = False  # Set to True to load saved model and skip training

if not SKIP_TRAINING:
    print("=" * 80)
    print("📚 TRAINING MODE")
    print("=" * 80)
    print("Continue running cells below to train a new model.")
    print("Skip this Quick Start section and proceed to Configuration.")
    print("=" * 80)

if SKIP_TRAINING:
    print("🚀 Loading pre-trained model from Google Drive...\n")

    # Install required packages
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'transformers', 'datasets', 'torch',
                   'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'openpyxl'],
                   check=False)

    import os
    import torch
    import json
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    # Model path
    MODEL_PATH = '/content/drive/MyDrive/thesis/emotion_classifier_phobertv2'

    # Load model and tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    # Load label mappings
    with open(os.path.join(MODEL_PATH, 'label_mappings.json'), 'r') as f:
        label_maps = json.load(f)
        label2id = label_maps['label2id']
        id2label = {int(k): v for k, v in label_maps['id2label'].items()}
        num_classes = len(id2label)

    # Set device (no TPU for inference after restart)
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"✓ Using device: cuda")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print(f"✓ Using device: cpu")

    model = model.to(device)
    USE_TPU = False

    print(f"✓ Model loaded from: {MODEL_PATH}")
    print(f"✓ Number of classes: {num_classes}")
    print(f"✓ Emotions: {list(id2label.values())}")
    print(f"\n✅ Ready for predictions! Jump to cell 'Predict on HuggingFace Dataset'")
    print("=" * 80)


## 📋 Configuration Section

**Adjust these hyperparameters based on your needs:**


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these hyperparameters as needed
# ============================================================================

# Random seed for reproducibility
SEED = 42

# Data paths - Using Google Drive (make sure your data is uploaded to this location)
DATA_DIR = '/content/drive/MyDrive/thesis/data'
MODEL_SAVE_PATH = '/content/drive/MyDrive/thesis/emotion_classifier_phobertv2'

# Model selection - PhoBERTv2 (135M parameters, trained on 140GB Vietnamese text)
MODEL_NAME = 'vinai/phobert-base-v2'  # PhoBERTv2 model for Vietnamese

# Batch size settings optimized for 20GB GPU RAM
# Note: Large model requires more memory - reduce batch size if OOM errors occur
PER_DEVICE_TRAIN_BATCH_SIZE = 16  # If you have enough GPU memory
PER_DEVICE_EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 4   # Effective batch size = 16 * 4 = 64

# Layer-wise Learning Rate Decay (LLRD) settings
BASE_LEARNING_RATE = 1.5e-6  # Base learning rate
LR_DECAY_FACTOR = 0.95  # Decay factor (0.65-0.95 recommended)
CLASSIFIER_LR_MULTIPLIER = 10.0  # Classifier head gets higher LR

# Training settings
NUM_EPOCHS = 20
WARMUP_RATIO = 0.1  # 10% of training steps for warmup
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 5

# Focal Loss settings
USE_FOCAL_LOSS = False  # Disabled - using standard CrossEntropyLoss
FOCAL_GAMMA = 2.0  # Not used when USE_FOCAL_LOSS is False
USE_CLASS_WEIGHTS = False  # Disabled - using uniform weights

# Data augmentation
USE_OVERSAMPLING = False  # Set to True to apply random oversampling

# Text preprocessing options
REMOVE_STOPWORDS = True  # Set to True to remove Vietnamese stopwords
USE_UNDERTHESEA_TOKENIZER = False  # Use PhoBERT's default BPE tokenizer

print("✓ Configuration loaded")
print(f"  Model: {MODEL_NAME}")
print(f"  Data dir: {DATA_DIR}")
print(f"  Model save path: {MODEL_SAVE_PATH}")
print(f"  Batch size (train): {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"  Batch size (eval): {PER_DEVICE_EVAL_BATCH_SIZE}")
print(f"  Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Base LR: {BASE_LEARNING_RATE:.2e}")
print(f"  Focal Loss: {USE_FOCAL_LOSS} (gamma={FOCAL_GAMMA if USE_FOCAL_LOSS else 'N/A'})")
print(f"  Class Weights: {USE_CLASS_WEIGHTS}")
print(f"  Oversampling: {USE_OVERSAMPLING}")
print(f"  Remove stopwords: {REMOVE_STOPWORDS}")
print(f"  Underthesea tokenizer: {USE_UNDERTHESEA_TOKENIZER}")
print(f"\n💡 GPU Memory Tips:")
print(f"  - If OOM error: reduce PER_DEVICE_TRAIN_BATCH_SIZE to 8 or 4")
print(f"  - With 20GB GPU: batch size 16 should work well")
print(f"  - FP16 mixed precision will be enabled automatically on GPU")


## 1. Install & Import Libraries


In [ ]:
%pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn openpyxl wandb


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              f1_score, precision_recall_fscore_support)
from sklearn.utils import resample
import torch
import torch.nn as nn
from torch.optim import AdamW
import json
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    get_linear_schedule_with_warmup
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check device - prioritize CUDA GPU, then CPU
USE_TPU = False

if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    print(f"✓ Using device: {device}")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    print(f"✓ Using device: {device} (Metal Performance Shaders)")
else:
    device = torch.device('cpu')
    print(f"✓ Using device: {device}")
    print("  ⚠️  Warning: No GPU detected. Training will be slower.")

print("✓ All libraries imported successfully")


## 📊 Weights & Biases (wandb) Setup

**Track your experiments with wandb for better visualization and comparison**

In [ ]:
import wandb
from google.colab import userdata

# Automatically retrieve wandb API key from Colab secrets
try:
    wandb_api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_api_key)
    print("✓ Successfully logged into wandb using Colab secrets")
except Exception as e:
    print(f"⚠️  Could not retrieve WANDB_API_KEY from Colab secrets: {e}")
    print("   Please add your wandb API key to Colab secrets:")
    print("   1. Click the 🔑 icon in the left sidebar")
    print("   2. Add a new secret with name: WANDB_API_KEY")
    print("   3. Paste your API key from https://wandb.ai/authorize")
    print("\n   Or login manually:")
    wandb.login()

# Initialize wandb project
wandb.init(
    project="emotion-classification-baseline",
    name=f"phobertv2-baseline",
    config={
        "model": "vinai/phobert-base-v2",
        "tokenizer": "phobert-default",
        "focal_loss": False,
        "class_weights": False,
        "experiment_type": "baseline"
    }
)
print("✓ wandb initialized")

## 2. Custom Trainer with LLRD

Implementing custom trainer with Layer-wise Learning Rate Decay for better fine-tuning


In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.
    
    Formula: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Class weights (tensor of shape [num_classes])
        gamma: Focusing parameter (default: 2.0)
        reduction: 'mean' or 'sum'
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # Class weights
        self.gamma = gamma  # Focusing parameter
        self.reduction = reduction
        
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Logits from model (batch_size, num_classes)
            targets: Ground truth labels (batch_size,)
        """
        # Calculate cross entropy
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        
        # Get probabilities
        p = torch.exp(-ce_loss)
        
        # Calculate focal loss
        focal_loss = (1 - p) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print("✓ Focal Loss class defined")


In [ ]:
class FocalLossTrainer(Trainer):
    """Custom Trainer that uses Focal Loss instead of Cross Entropy."""
    
    def __init__(self, *args, focal_loss_fn=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss_fn = focal_loss_fn
        
    def compute_loss(self, model, inputs, return_outputs=False):
        """
        Override compute_loss to use Focal Loss.
        """
        labels = inputs.pop("labels")
        
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Compute focal loss
        if self.focal_loss_fn is not None:
            loss = self.focal_loss_fn(logits, labels)
        else:
            # Fallback to standard cross entropy
            loss = nn.functional.cross_entropy(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("✓ FocalLossTrainer class defined")

## 3. Layer-wise Learning Rate Decay (LLRD) Implementation

**Key Concept:** Different layers get different learning rates
- **Classifier head:** Highest LR (3.5e-5 = 10× base)
- **Top encoder layers:** Base LR (3.5e-6)  
- **Middle layers:** Gradually decreasing  
- **Embeddings:** Lowest LR (~1.9e-6)

**Formula:** `LR_layer_i = LR_base × (decay_factor)^(num_layers - i)`

**Why?** Lower layers encode general linguistic knowledge → preserve it with lower LR


In [ ]:
def get_optimizer_grouped_parameters(model, base_lr=1.5e-6, lr_decay_factor=0.95,
                                    weight_decay=0.01, classifier_lr_multiplier=10.0):
    """Create parameter groups with layer-wise learning rate decay."""

    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    optimizer_grouped_parameters = []

    # Get model type - PhoBERTv2 uses RoBERTa architecture
    if hasattr(model, 'roberta'):
        encoder = model.roberta
        model_type = 'roberta'
    elif hasattr(model, 'bert'):
        encoder = model.bert
        model_type = 'bert'
    else:
        raise ValueError("Model type not supported for LLRD")

    num_layers = len(encoder.encoder.layer)

    print(f"\n{'='*80}")
    print("LAYER-WISE LEARNING RATE DECAY (LLRD)")
    print(f"{'='*80}")
    print(f"Model: {model_type} | Layers: {num_layers} | Base LR: {base_lr:.2e} | Decay: {lr_decay_factor}")
    print("\nLearning rates by layer:")

    # 1. Classifier head (highest LR)
    classifier_lr = base_lr * classifier_lr_multiplier
    optimizer_grouped_parameters.extend([
        {'params': [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': classifier_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': classifier_lr, 'weight_decay': 0.0}
    ])
    print(f"  Classifier head: {classifier_lr:.2e}")

    # 2. Encoder layers (top to bottom with decay)
    for layer_idx in range(num_layers - 1, -1, -1):
        layer = encoder.encoder.layer[layer_idx]
        lr = base_lr * (lr_decay_factor ** (num_layers - 1 - layer_idx))

        optimizer_grouped_parameters.extend([
            {'params': [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': weight_decay},
            {'params': [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': 0.0}
        ])

        if layer_idx % 3 == 0:
            print(f"  Layer {layer_idx}: {lr:.2e}")

    # 3. Embeddings (lowest LR)
    embedding_lr = base_lr * (lr_decay_factor ** num_layers)
    optimizer_grouped_parameters.extend([
        {'params': [p for n, p in encoder.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': embedding_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in encoder.embeddings.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': embedding_lr, 'weight_decay': 0.0}
    ])
    print(f"  Embeddings: {embedding_lr:.2e}")
    print(f"{'='*80}\n")

    return optimizer_grouped_parameters

print("✓ LLRD function defined")


In [ ]:
class AdvancedTrainer(Trainer):
    """Advanced Trainer with LLRD + Focal Loss Support."""

    def __init__(self, *args, llrd_config=None, focal_loss_fn=None, **kwargs):
        self.llrd_config = llrd_config or {}
        self.focal_loss_fn = focal_loss_fn
        super().__init__(*args, **kwargs)

    def create_optimizer(self):
        """Create optimizer with LLRD parameter groups."""
        if self.optimizer is None:
            optimizer_grouped_parameters = get_optimizer_grouped_parameters(
                self.model,
                base_lr=self.llrd_config.get('base_lr', 1.5e-6),
                lr_decay_factor=self.llrd_config.get('lr_decay_factor', 0.95),
                weight_decay=self.args.weight_decay,
                classifier_lr_multiplier=self.llrd_config.get('classifier_lr_multiplier', 10.0)
            )

            self.optimizer = AdamW(optimizer_grouped_parameters, betas=(0.9, 0.999), eps=1e-8)

        return self.optimizer

    def create_scheduler(self, num_training_steps: int, optimizer=None):
        """Create LR scheduler. CRITICAL: Steps after every batch!"""
        if self.lr_scheduler is None:
            warmup_steps = int(num_training_steps * self.args.warmup_ratio)

            self.lr_scheduler = get_linear_schedule_with_warmup(
                optimizer=self.optimizer if optimizer is None else optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=num_training_steps
            )

            print(f"\n✓ Scheduler created: {num_training_steps} steps, {warmup_steps} warmup")
            print("  ⚠️  CRITICAL: Scheduler steps after EVERY batch\n")

        return self.lr_scheduler
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Override compute_loss to use Focal Loss if provided.
        Compatible with newer transformers versions that pass num_items_in_batch.
        """
        labels = inputs.pop("labels")
        
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Compute loss (Focal Loss or Cross Entropy)
        if self.focal_loss_fn is not None:
            loss = self.focal_loss_fn(logits, labels)
        else:
            loss = nn.functional.cross_entropy(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

print("✓ AdvancedTrainer class defined")


## 4. Load and Prepare Data

Loading NEU-ESC dataset and preparing for training

### Text Preprocessing Features:
- **Vietnamese text standardization**: Normalizes Unicode, handles URLs, emails, mentions, hashtags
- **Slang normalization**: Converts common Vietnamese text slang to standard forms
- **Stopword removal** (configurable): Removes common Vietnamese stopwords that may not contribute to emotion classification
  - Articles, pronouns, prepositions, common conjunctions
  - 60+ common Vietnamese stopwords
  - Can be enabled/disabled via `REMOVE_STOPWORDS` config


In [ ]:
# Vietnamese stopwords list
VIETNAMESE_STOPWORDS = {
    # Articles & determiners
    "các", "của", "cho", "và", "trong", "một", "là", "được", "đã", "có", "này", 
    "đó", "những", "để", "từ", "với", "về", "cũng", "như", "còn", "nhưng", "hay",
    # Pronouns
    "tôi", "bạn", "anh", "chị", "em", "chúng", "ta", "mình", "họ", "nó", "ai",
    # Prepositions & conjunctions
    "ở", "trên", "dưới", "ngoài", "sau", "trước", "giữa", "bên", "cùng", "theo",
    "đến", "khi", "nếu", "mà", "thì", "vì", "do", "bởi", "nên",
    # Common verbs & auxiliaries
    "rằng", "thế", "vậy", "thôi", "đâu", "sao", "gì", "nào", "đây", "đấy", "kia",
    # Quantifiers
    "nhiều", "ít", "vài", "mấy", "bao", "lắm",
    # Others
    "rất", "quá", "khá", "hơi", "tất", "mọi", "khác", "cả", "nữa", "luôn", "đều"
}

# Vietnamese text standardization and stopword removal (import with safe fallback)
try:
    from tm_research.text_preprocess import standardize_vietnamese_text, remove_vietnamese_stopwords
    print("✓ Loaded text preprocessing functions from tm_research.text_preprocess")
except Exception:
    import re, unicodedata
    _URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
    _EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
    _USER_RE = re.compile(r"@(\w+)")
    _HASH_RE = re.compile(r"#(\w+)")
    _MULTI_SPACE_RE = re.compile(r"\s+")
    _REPEAT_CHAR_RE = re.compile(r"(.)\1{2,}")
    _PUNCT_SPACING_RE = re.compile(r"\s*([,.!?;:()\[\]{}…])\s*")
    _VI_SLANG_MAP = {"k":"không","ko":"không","kh":"không","hok":"không","hông":"không","hk":"không","j":"gì","dc":"được","đc":"được","vl":"vãi","vcl":"vãi","iu":"yêu","thik":"thích","oke":"ok","okie":"ok", "jz":"gì vậy"}
    
    def standardize_vietnamese_text(text, remove_stopwords=False):
        """Standardize Vietnamese text with optional stopword removal."""
        if text is None:
            return ""
        t = unicodedata.normalize("NFC", str(text)).strip().lower()
        t = _URL_RE.sub(" <url> ", t)
        t = _EMAIL_RE.sub(" <email> ", t)
        t = _USER_RE.sub(lambda m: f" <user:{m.group(1)}> ", t)
        t = _HASH_RE.sub(lambda m: f" <hashtag:{m.group(1)}> ", t)
        t = _REPEAT_CHAR_RE.sub(r"\1\1", t)
        t = _PUNCT_SPACING_RE.sub(r" \1 ", t)
        toks = [ _VI_SLANG_MAP.get(tok, tok) for tok in t.split() ]
        
        # Remove stopwords if requested
        if remove_stopwords:
            toks = [tok for tok in toks if tok not in VIETNAMESE_STOPWORDS]
        
        t = " ".join(toks)
        t = _MULTI_SPACE_RE.sub(" ", t).strip()
        return t
    
    def remove_vietnamese_stopwords(text):
        """Remove Vietnamese stopwords from text."""
        return standardize_vietnamese_text(text, remove_stopwords=True)
    
    print("ℹ️  Fallback text preprocessing functions defined (with stopword removal)")

# Underthesea word segmentation for Vietnamese
_underthesea_available = False
if USE_UNDERTHESEA_TOKENIZER:
    try:
        from underthesea import word_tokenize as uts_word_tokenize
        _underthesea_available = True
        print("✓ Underthesea word tokenizer loaded")
    except ImportError:
        print("⚠️  Underthesea not available, falling back to simple tokenization")

def tokenize_vietnamese(text, remove_stopwords=False):
    """Tokenize Vietnamese text using underthesea word segmentation.
    
    Vietnamese words can be multi-syllable (e.g., 'học sinh' = student).
    Underthesea segments text into proper Vietnamese words.
    """
    if text is None:
        return ""
    
    # First apply standardization
    t = standardize_vietnamese_text(text, remove_stopwords=False)
    
    # Apply underthesea word segmentation if available
    if _underthesea_available:
        t = uts_word_tokenize(t, format='text')  # Returns text with underscores for compound words
    
    # Remove stopwords after tokenization if requested
    if remove_stopwords:
        toks = t.split()
        toks = [tok for tok in toks if tok.replace('_', ' ') not in VIETNAMESE_STOPWORDS and tok not in VIETNAMESE_STOPWORDS]
        t = ' '.join(toks)
    
    return t

print(f"✓ Vietnamese text preprocessing ready")
print(f"✓ Loaded {len(VIETNAMESE_STOPWORDS)} Vietnamese stopwords")
if USE_UNDERTHESEA_TOKENIZER and _underthesea_available:
    print(f"✓ Underthesea word segmentation enabled")


In [ ]:
# Create model save directory if it doesn't exist
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
print(f"✓ Model save directory ready: {MODEL_SAVE_PATH}")

# Load datasets
train_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/train_processed.csv'))
val_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/val_processed.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/test_processed.csv'))

print(f"\n✓ Datasets loaded from Google Drive")
print(f"  Train: {train_df.shape[0]} samples")
print(f"  Val: {val_df.shape[0]} samples")
print(f"  Test: {test_df.shape[0]} samples")

print(train_df.columns)
print(val_df.columns)
print(test_df.columns)

train_df = train_df.drop(["Emotion", "text_length","word_count"], axis=1)
val_df = val_df.drop(["Emotion"],axis = 1)
test_df = test_df.drop(["Emotion"],axis=1)


train_df.columns = ['text', 'label']
val_df.columns = ['text', 'label']
test_df.columns = ['text', 'label']

# Clean data
for df in [train_df, val_df, test_df]:
    df.dropna(subset=['text', 'label'], inplace=True)
    df['text'] = df['text'].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip()
    
    # Vietnamese text preprocessing with underthesea tokenization and optional stopword removal
    if USE_UNDERTHESEA_TOKENIZER:
        df['text'] = df['text'].apply(lambda x: tokenize_vietnamese(x, remove_stopwords=REMOVE_STOPWORDS))
        print(f"✓ Applied underthesea tokenization{' + stopword removal' if REMOVE_STOPWORDS else ''} to {df.shape[0]} samples")
    elif REMOVE_STOPWORDS:
        df['text'] = df['text'].apply(lambda x: standardize_vietnamese_text(x, remove_stopwords=True))
        print(f"✓ Applied standardization + stopword removal to {df.shape[0]} samples")
    else:
        df['text'] = df['text'].apply(standardize_vietnamese_text)
        print(f"✓ Applied standardization (no stopword removal) to {df.shape[0]} samples")

# Show sample before/after
print(f"\n{'='*80}")
print("SAMPLE TEXT AFTER PREPROCESSING")
print(f"{'='*80}")
print(f"Sample 1: {train_df['text'].iloc[0][:150]}...")
print(f"Sample 2: {train_df['text'].iloc[1][:150]}...")
print(f"Sample 3: {train_df['text'].iloc[2][:150]}...")
print(f"{'='*80}")

if REMOVE_STOPWORDS:
    print("\n✓ Data cleaned, standardized, and stopwords removed")
else:
    print("\n✓ Data cleaned and standardized (stopwords retained)")


## 5. Analyze Class Distribution

Understanding the class distribution in the training data


In [ ]:
# Analyze distribution
label_counts = train_df['label'].value_counts()
print("Emotion distribution (Training set):")
print(label_counts)
print(f"\nImbalance ratio: {label_counts.max() / label_counts.min():.1f}:1")

# Visualize
plt.figure(figsize=(12, 6))
label_counts.sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Emotion', fontsize=12, fontweight='bold')
plt.ylabel('Count', fontsize=12, fontweight='bold')
plt.title('Training Set - Emotion Distribution', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Create label mappings
unique_labels = sorted(train_df['label'].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_classes = len(unique_labels)

# Add numeric labels
train_df['labels'] = train_df['label'].map(label2id)
val_df['labels'] = val_df['label'].map(label2id)
test_df['labels'] = test_df['label'].map(label2id)

print(f"\n✓ Label mappings created ({num_classes} classes)")

# Calculate class weights for Focal Loss using ORIGINAL distribution
# Original distribution (before augmentation):
#   Enjoyment: 1558, Disgust: 1071, Other: 1021, Sadness: 947
#   Anger: 391, Fear: 318, Surprise: 242
if USE_CLASS_WEIGHTS:
    # Use original class distribution for weight calculation
    original_distribution = {
        'buồn': 947,          # Sadness
        'giận dữ': 391,       # Anger
        'khác': 1021,         # Other
        'khó chịu': 1071,     # Disgust
        'ngạc nhiên': 242,    # Surprise
        'sợ hãi': 318,        # Fear
        'vui vẻ': 1558        # Enjoyment
    }
    
    original_total = sum(original_distribution.values())  # 5548
    class_weights = []
    
    print(f"\n{'='*80}")
    print("CLASS WEIGHTS FOR FOCAL LOSS (Based on Original Distribution)")
    print(f"{'='*80}")
    print(f"Original total samples: {original_total}")
    print(f"Current training samples: {len(train_df)} (augmented)")
    print()
    
    for label in unique_labels:
        label_id = label2id[label]
        original_count = original_distribution.get(label, label_counts[label])
        # Inverse frequency: total / (num_classes * count)
        weight = original_total / (num_classes * original_count)
        class_weights.append((label_id, weight, original_count))
        current_count = label_counts.get(label, 0)
        print(f"{label:20s}: weight={weight:.4f} (original: {original_count}, current: {current_count})")
    
    # Sort by label_id to ensure correct order
    class_weights.sort(key=lambda x: x[0])
    class_weights_tensor = torch.tensor([w[1] for w in class_weights], dtype=torch.float32)
    
    print(f"{'='*80}\n")
    print(f"✓ Class weights calculated from original distribution")
    print(f"  Weights: {class_weights_tensor.numpy()}")
else:
    class_weights_tensor = None
    print(f"\n✓ Class weights disabled (using uniform weights)")


## 6. Tokenize Data & Load Model


In [ ]:
# Convert to Dataset format
train_dataset = Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True))

print(f"✓ Datasets converted to HuggingFace format")

# Load model and tokenizer
print(f"\nLoading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)

print(f"✓ Model and tokenizer loaded")
print(f"✓ Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

print(f"\n✓ Tokenized {len(train_tokenized)} training samples")
print(f"✓ Tokenized {len(val_tokenized)} validation samples")
print(f"✓ Tokenized {len(test_tokenized)} test samples")


## 7. Train Advanced Model with LLRD + Focal Loss

Training with:
- ✅ Layer-wise Learning Rate Decay (LLRD)  
- ✅ Focal Loss with class weights (for imbalanced data)
- ✅ Gradient accumulation  
- ✅ Early stopping


In [ ]:
# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='./results_advanced',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=BASE_LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    logging_dir='./logs_advanced',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',

    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    save_total_limit=2,

    # TPU/GPU settings for Colab
    tpu_num_cores=8 if USE_TPU else None,  # Use 8 TPU cores if available
    fp16=(device.type == 'cuda'),  # Enable mixed precision on GPU
    dataloader_num_workers=0,
    seed=SEED,
)

# Move model to device
model = model.to(device)

# Initialize Focal Loss if enabled
if USE_FOCAL_LOSS:
    # Move class weights to device if using them
    if class_weights_tensor is not None:
        class_weights_device = class_weights_tensor.to(device)
    else:
        class_weights_device = None
    
    focal_loss_fn = FocalLoss(
        alpha=class_weights_device,
        gamma=FOCAL_GAMMA,
        reduction='mean'
    )
    print(f"✓ Focal Loss initialized (gamma={FOCAL_GAMMA}, class_weights={'enabled' if class_weights_device is not None else 'disabled'})")
else:
    focal_loss_fn = None
    print(f"✓ Using standard Cross Entropy Loss")

# LLRD configuration
llrd_config = {
    'base_lr': BASE_LEARNING_RATE,
    'lr_decay_factor': LR_DECAY_FACTOR,
    'classifier_lr_multiplier': CLASSIFIER_LR_MULTIPLIER
}

# Create trainer
trainer = AdvancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],

    # LLRD parameters
    llrd_config=llrd_config,
    # Focal Loss
    focal_loss_fn=focal_loss_fn
)

print(f"\n{'='*80}")
print("TRAINING CONFIGURATION")
print(f"{'='*80}")
print(f"Effective batch size: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Device: {device}")
if USE_FOCAL_LOSS:
    print(f"Loss function: Focal Loss (gamma={FOCAL_GAMMA})")
    if class_weights_tensor is not None:
        print(f"  - Class weights: enabled")
    else:
        print(f"  - Class weights: disabled (uniform)")
else:
    print(f"Loss function: Cross Entropy (standard)")
print(f"{'='*80}\n")

print("✓ Trainer initialized with:")
print("  - Layer-wise Learning Rate Decay (LLRD)")
if USE_FOCAL_LOSS:
    print(f"  - Focal Loss (gamma={FOCAL_GAMMA})")
    if class_weights_tensor is not None:
        print(f"  - Class weights for imbalance handling")
else:
    print("  - Cross Entropy Loss")
print("  - Gradient accumulation")
print("  - Early stopping")


In [ ]:
# START TRAINING
print("\n🚀 Starting training...\n")
train_result = trainer.train()

print(f"\n{'='*80}")
print("✅ TRAINING COMPLETED!")
print(f"{'='*80}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Samples/second: {train_result.metrics['train_samples_per_second']:.2f}")
print(f"{'='*80}\n")


In [ ]:
# After training completes
import matplotlib.pyplot as plt

# Plot training history
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_loss, label='Training Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss Over Time')

plt.subplot(1, 2, 2)
plt.plot(eval_loss, label='Validation Loss', color='orange')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Validation Loss Over Time')
plt.tight_layout()
plt.show()

## 8. Save Model to Google Drive

Save the trained model to your Google Drive for later use


In [ ]:
# Save the best model and tokenizer to Google Drive
print(f"💾 Saving model to Google Drive...")
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

# Save label mappings for later use
import json
label_mappings = {
    'label2id': label2id,
    'id2label': id2label
}
with open(os.path.join(MODEL_SAVE_PATH, 'label_mappings.json'), 'w', encoding='utf-8') as f:
    json.dump(label_mappings, f, ensure_ascii=False, indent=2)

print(f"✓ Model saved successfully to: {MODEL_SAVE_PATH}")
print(f"  - Model weights")
print(f"  - Tokenizer")
print(f"  - Label mappings")
print(f"\n📁 You can access this model from your Google Drive anytime!")

# Finish wandb run
wandb.finish()
print("✓ wandb run finished")

# Auto timeout to save Colab resources
print("\n" + "="*80)
print("🔌 AUTO TIMEOUT: Disconnecting runtime to save resources...")
print("="*80)
print("Training complete! The runtime will disconnect in 60 seconds.")
print("Your model is safely saved to Google Drive.")

import time
time.sleep(60)

# Disconnect runtime
from google.colab import runtime
runtime.unassign()
